# Hand-Gesture Detection + Segmentation with YOLOv1

This notebook trains ONE multi-task network that, for every RGB image:
1. **classifies** which of the 10 gestures (`G01_call` ... `G10_three`) is shown,
2. **localises** it with a bounding box (classic **YOLOv1** grid formulation),
3. **segments** the hand with a pixel mask (U-Net-style decoder on the shared backbone).

**Why this architecture?** YOLOv1 (Redmon et al., 2016) predicts boxes and classes in a single forward pass over an S x S grid, but it has no segmentation output. Instead of training a second network, we attach a small U-Net-style decoder to the same backbone: both tasks need the same hand features, so sharing them halves compute and regularises a tiny dataset.

**How to run:** `Runtime -> Change runtime type -> GPU`, then run all cells top to bottom. Upload `dataset_merged_all.zip` when the upload widget appears.

In [ ]:
# ---------------- Step 0: imports + reproducibility ----------------
import os, json, random, zipfile          # stdlib: paths, annotation parsing, seeding, unzipping
import numpy as np                        # array math for masks, boxes and metrics
import torch                              # tensors, autograd, GPU
import torch.nn as nn                     # network layers
import torch.nn.functional as F           # interpolate / losses
from PIL import Image, ImageDraw          # image IO + drawing (lighter than OpenCV, preinstalled)
from torch.utils.data import Dataset, DataLoader   # data pipeline
from torchvision import transforms                  # colour jitter + ImageNet normalisation
from torchvision.models import resnet18, ResNet18_Weights  # pretrained backbone
import matplotlib.pyplot as plt          # curves and visualisations
from tqdm.auto import tqdm               # progress bars

SEED = 42                                 # one fixed seed everywhere -> reproducible runs
random.seed(SEED)                         # python RNG (augmentation coin flips, split)
np.random.seed(SEED)                      # numpy RNG
torch.manual_seed(SEED)                   # torch CPU RNG
torch.cuda.manual_seed_all(SEED)          # torch GPU RNG
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'  # use the Colab GPU when available
print('device:', DEVICE)                  # sanity check: should print cuda on Colab GPU runtime


## Step 1 — get the dataset
Upload `dataset_merged_all.zip` (Option A) or copy it from Google Drive (Option B). We unzip with Python's `zipfile` instead of a shell command so the cell also works outside Colab.

In [ ]:
# ---------------- Option A: direct upload (simplest) ----------------
if not os.path.exists('dataset_merged_all.zip') and not os.path.exists('data'):
    try:                                   # google.colab only exists on Colab, so we guard the import
        from google.colab import files     # Colab file-upload widget
        files.upload()                     # pick dataset_merged_all.zip in the dialog
    except ImportError:                    # running locally instead
        print('Not on Colab: place dataset_merged_all.zip next to this notebook.')

# ---------------- Option B: Google Drive (uncomment if the zip is on Drive) ----------------
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil; shutil.copy('/content/drive/MyDrive/dataset_merged_all.zip', '.')

if not os.path.exists('data'):             # skip if already extracted (re-running is cheap)
    with zipfile.ZipFile('dataset_merged_all.zip') as zf:  # pure-python unzip: portable
        zf.extractall('data')              # extract into ./data regardless of zip-internal layout
print('dataset ready under ./data')


## Step 2 — explore the dataset
We *walk* the extracted tree to find the class folders instead of hard-coding paths, because zips often extract with an extra top-level directory. A class folder is any folder that contains both an `rgb` and an `annotation` subfolder. This cell also auto-detects the annotation file format — the loader below adapts to mask images, JSON, or txt boxes.

In [ ]:
IMG_EXTS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}   # extensions treated as images

def find_class_dirs(root):
    dirs = []                                                    # collected class folders
    for dirpath, dirnames, _ in os.walk(root):                   # walk the whole extracted tree
        lower = {d.lower() for d in dirnames}                    # case-insensitive matching
        if any('rgb' in d for d in lower) and any('annot' in d for d in lower):
            dirs.append(dirpath)                                 # has rgb + annotation -> class dir
    return sorted(dirs)                                          # sorted -> stable class indices

class_dirs = find_class_dirs('data')                             # locate G01_call ... G10_three
CLASS_NAMES = [os.path.basename(d).split('_', 1)[-1] for d in class_dirs]  # 'G01_call' -> 'call'
print('classes:', CLASS_NAMES)                                   # expect the 10 gesture names

for d in class_dirs:                                             # show what each class folder holds
    subs = {s.lower(): os.path.join(d, s) for s in os.listdir(d)
            if os.path.isdir(os.path.join(d, s))}                # its subfolders (rgb/depth/annotation)
    counts = {name: len(os.listdir(p)) for name, p in subs.items()}  # file count per subfolder
    print(os.path.basename(d), counts)

ann_dir = next(p for n, p in {s.lower(): os.path.join(class_dirs[0], s)
               for s in os.listdir(class_dirs[0])}.items() if 'annot' in n)  # first annotation folder
ann_ext = os.path.splitext(sorted(os.listdir(ann_dir))[0])[1].lower()        # peek at its file type
print('annotation format detected:', ann_ext)                    # the loader adapts to this


## Step 3 — configuration
All hyper-parameters in one place so every later cell reads from a single source of truth. `S=7, B=2` are the original YOLOv1 paper values; `IMG_SIZE=448` is YOLOv1's native resolution and equals `7 * 64`, so the backbone's /32 feature map (14x14) becomes exactly 7x7 after one stride-2 conv.

In [ ]:
S = 7               # YOLOv1 grid size (paper value; one hand per image needs no finer grid)
B = 2               # boxes per cell (paper value; lets boxes specialise wide vs tall)
C = len(CLASS_NAMES)  # number of gesture classes, read from the data itself (should be 10)
IMG_SIZE = 448      # YOLOv1 native input resolution (448 = S * 64)
BATCH_SIZE = 8      # small dataset + small batch fits any Colab GPU
EPOCHS = 60         # enough to plateau on ~60 images without wasting GPU time
LR = 1e-4           # small LR: we are fine-tuning a pretrained backbone, not training from scratch
WEIGHT_DECAY = 1e-4 # mild L2 regularisation, important on a tiny dataset
LAMBDA_COORD = 5.0  # paper value: boost the box-coordinate loss (localisation is hardest)
LAMBDA_NOOBJ = 0.5  # paper value: damp empty-cell confidence loss (48/49 cells are empty)
LAMBDA_SEG = 2.0    # weight of segmentation vs detection loss (balances the two tasks)
VAL_FRACTION = 0.2  # 20% of EACH class held out for validation
IMAGENET_MEAN = [0.485, 0.456, 0.406]  # backbone was pretrained with these statistics,
IMAGENET_STD = [0.229, 0.224, 0.225]   # so inputs must be normalised the same way


## Step 4 — annotation loading (format-adaptive) + box helpers
The bounding box is **derived from the segmentation mask** (`mask_to_bbox`): a tight box computed from the mask can never disagree with the mask, so the two supervision signals stay consistent by construction. If the annotations turn out to be JSON polygons or txt boxes instead of mask images, the same function transparently handles those too.

In [ ]:
def digits_key(filename):
    # rgb_003.png and mask_003.png share no stem but DO share the digits 003 -> best pairing key
    stem = os.path.splitext(os.path.basename(filename))[0]       # drop folder + extension
    digits = ''.join(ch for ch in stem if ch.isdigit())          # keep only 0-9
    return digits if digits else stem                            # fall back to the full stem

def mask_to_bbox(mask):
    ys, xs = np.nonzero(mask)                                    # foreground pixel coordinates
    if len(xs) == 0:                                             # empty mask (bad annotation):
        h, w = mask.shape                                        # fall back to a full-image box
        return [0, 0, w - 1, h - 1]                              # so training never crashes
    return [int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())]  # tight box

def _find_in_json(obj, keys):
    # recursive search: annotation JSONs are often nested and we do not know the schema
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k.lower() in keys:                                # case-insensitive key match
                return v
        for v in obj.values():                                   # descend into every value
            r = _find_in_json(v, keys)
            if r is not None:
                return r
    elif isinstance(obj, list):
        for v in obj:                                            # descend into every element
            r = _find_in_json(v, keys)
            if r is not None:
                return r
    return None

def load_annotation(path, img_hw):
    # returns (binary mask HxW uint8, bbox [x1,y1,x2,y2] pixels) for ANY supported format
    h, w = img_hw                                                # target size = RGB image size
    ext = os.path.splitext(path)[1].lower()                      # pick the parser by extension
    if ext in IMG_EXTS:                                          # case 1: the annotation IS a mask image
        mask = np.array(Image.open(path).convert('L'))           # force single-channel grayscale
        if mask.shape != (h, w):                                 # mask may be a different size
            mask = np.array(Image.fromarray(mask).resize((w, h), Image.NEAREST))  # NEAREST keeps it binary
        mask = (mask > 0).astype(np.uint8)                       # any non-zero pixel = hand
        return mask, mask_to_bbox(mask)                          # box derived from the mask
    if ext == '.json':                                           # case 2: JSON annotation
        with open(path) as f:
            data = json.load(f)                                  # parse the document
        pts = _find_in_json(data, {'points', 'polygon', 'segmentation', 'landmarks'})
        if pts is not None:                                      # polygon found -> rasterise it
            pts = np.array(pts, dtype=np.float32).reshape(-1, 2) # normalise to (N,2)
            if pts.max() <= 1.5:                                 # values look normalised (0..1)
                pts = pts * np.array([w, h], dtype=np.float32)   # scale to pixels
            canvas = Image.new('L', (w, h), 0)                   # blank mask canvas
            ImageDraw.Draw(canvas).polygon([tuple(p) for p in pts], fill=1)  # fill polygon
            mask = np.array(canvas, dtype=np.uint8)
            return mask, mask_to_bbox(mask)
        box = _find_in_json(data, {'bbox', 'box', 'rect', 'bounding_box'})
        if box is not None:                                      # only a box -> rectangle mask proxy
            b = [float(v) for v in box]
            if max(b) <= 1.5:                                    # normalised -> pixels
                b = [b[0] * w, b[1] * h, b[2] * w, b[3] * h]
            if b[2] <= b[0] or b[3] <= b[1] or (b[0] + b[2] <= w and b[2] < w * 0.9):
                b = [b[0], b[1], b[0] + b[2], b[1] + b[3]]       # heuristic: x,y,w,h -> corners
            bbox = [int(max(0, b[0])), int(max(0, b[1])), int(min(w - 1, b[2])), int(min(h - 1, b[3]))]
            mask = np.zeros((h, w), dtype=np.uint8)              # a rectangle is the best seg proxy
            mask[bbox[1]:bbox[3] + 1, bbox[0]:bbox[2] + 1] = 1   # a bare box can give us
            return mask, bbox
        raise ValueError('Unrecognised JSON annotation schema: ' + path)
    if ext in {'.txt', '.csv'}:                                  # case 3: plain-text box
        toks = open(path).read().replace(',', ' ').split()       # all whitespace/comma tokens
        nums = [float(t) for t in toks if t.replace('.', '', 1).replace('-', '', 1).isdigit()]
        if len(nums) >= 5:                                       # YOLO txt: class cx cy w h (normalised)
            _, cx, cy, bw, bh = nums[:5]
            bbox = [int((cx - bw / 2) * w), int((cy - bh / 2) * h), int((cx + bw / 2) * w), int((cy + bh / 2) * h)]
        elif len(nums) == 4:                                     # bare box: assume x1 y1 x2 y2
            bbox = [int(v) for v in nums]
        else:
            raise ValueError('Cannot parse text annotation: ' + path)
        bbox = [max(0, bbox[0]), max(0, bbox[1]), min(w - 1, bbox[2]), min(h - 1, bbox[3])]
        mask = np.zeros((h, w), dtype=np.uint8)                  # rectangle mask proxy again
        mask[bbox[1]:bbox[3] + 1, bbox[0]:bbox[2] + 1] = 1
        return mask, bbox
    raise ValueError('Unsupported annotation format: ' + path)   # fail loudly on unknown formats

def iou_xyxy(a, b):
    # standard box IoU used for evaluation
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])                  # intersection top-left
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])                  # intersection bottom-right
    inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)            # clamp: disjoint boxes -> 0
    area_a = max(0.0, a[2] - a[0]) * max(0.0, a[3] - a[1])       # area of a
    area_b = max(0.0, b[2] - b[0]) * max(0.0, b[3] - b[1])       # area of b
    union = area_a + area_b - inter                              # inclusion-exclusion
    return inter / union if union > 0 else 0.0                   # guard divide-by-zero


## Step 5 — dataset, per-class split and YOLO target encoding
Notes on the choices here:
* **Per-class split** instead of a plain random split: with only ~6 images per class, a random 80/20 split could leave a class with zero validation images.
* **Depth images are not used** in this baseline: the pretrained ResNet expects 3-channel RGB, and with so few images the ImageNet-pretrained features matter far more than a 4th channel trained from scratch (README describes how to add depth later).
* **Only horizontal flips + colour jitter** as augmentation: flips preserve gesture identity for these classes, and colour jitter changes no geometry, so boxes/masks stay valid. Rotations/crops would need box re-computation and can cut off fingers that define the gesture.

In [ ]:
def scan_dataset(dirs):
    samples = []                                                 # one record per rgb image
    for cls_idx, cdir in enumerate(dirs):                        # class index = position in sorted list
        subs = {s.lower(): os.path.join(cdir, s) for s in os.listdir(cdir)
                if os.path.isdir(os.path.join(cdir, s))}         # subfolders of this class
        rgb_dir = next(p for n, p in subs.items() if 'rgb' in n)      # locate rgb folder
        ann_dir = next(p for n, p in subs.items() if 'annot' in n)    # locate annotation folder
        rgb_files = sorted(f for f in os.listdir(rgb_dir)
                           if os.path.splitext(f)[1].lower() in IMG_EXTS)  # image files, sorted
        ann_files = sorted(os.listdir(ann_dir))                  # annotations, any extension
        ann_by_key = {digits_key(f): f for f in ann_files}       # pair by shared digits
        for pos, rf in enumerate(rgb_files):
            af = ann_by_key.get(digits_key(rf))                  # digits pairing first
            if af is None and pos < len(ann_files):              # positional fallback (both sorted)
                af = ann_files[pos]
            samples.append({'rgb': os.path.join(rgb_dir, rf),
                            'ann': os.path.join(ann_dir, af),
                            'cls': cls_idx})
    return samples

def split_per_class(samples, val_fraction, seed):
    rng = random.Random(seed)                                    # local RNG -> reproducible split
    by_class = {}
    for idx, s in enumerate(samples):                            # group indices by class
        by_class.setdefault(s['cls'], []).append(idx)
    train_idx, val_idx = [], []
    for cls, idxs in by_class.items():
        rng.shuffle(idxs)                                        # shuffle within the class
        n_val = max(1, int(round(len(idxs) * val_fraction)))     # AT LEAST one val image per class
        val_idx += idxs[:n_val]
        train_idx += idxs[n_val:]
    return train_idx, val_idx

def encode_yolo_target(bbox, cls_idx):
    # one GT box -> (S,S,5+C) grid: [x_cell, y_cell, w, h, obj, one-hot classes]
    t = torch.zeros(S, S, 5 + C)                                 # all cells empty by default
    x1, y1, x2, y2 = bbox
    cx = (x1 + x2) / 2 / IMG_SIZE                                # normalised centre x in [0,1]
    cy = (y1 + y2) / 2 / IMG_SIZE                                # normalised centre y in [0,1]
    w = (x2 - x1) / IMG_SIZE                                     # normalised width
    h = (y2 - y1) / IMG_SIZE                                     # normalised height
    j = min(S - 1, int(cx * S))                                  # grid column holding the centre
    i = min(S - 1, int(cy * S))                                  # grid row holding the centre
    t[i, j, 0] = cx * S - j                                      # centre x RELATIVE to its cell:
    t[i, j, 1] = cy * S - i                                      # this keeps coords in [0,1] (YOLO trick)
    t[i, j, 2] = w                                               # w,h stay image-relative
    t[i, j, 3] = h
    t[i, j, 4] = 1.0                                             # objectness: this cell owns the object
    t[i, j, 5 + cls_idx] = 1.0                                   # one-hot class
    return t

class GestureDataset(Dataset):
    def __init__(self, samples, indices, augment):
        self.items = [samples[i] for i in indices]               # only this split's samples
        self.augment = augment                                   # augment only the training split
        self.jitter = transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3)
        self.normalize = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)  # match the backbone
    def __len__(self):
        return len(self.items)
    def __getitem__(self, idx):
        item = self.items[idx]
        img = Image.open(item['rgb']).convert('RGB')             # force 3 channels (some pngs are RGBA)
        w0, h0 = img.size                                        # original size, needed to scale the box
        mask, bbox = load_annotation(item['ann'], (h0, w0))      # format-adaptive load
        img = img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)   # smooth resize is fine for photos
        mask = Image.fromarray(mask * 255).resize((IMG_SIZE, IMG_SIZE), Image.NEAREST)
        #      NEAREST for masks: bilinear would invent fractional labels between 0 and 1
        sx, sy = IMG_SIZE / w0, IMG_SIZE / h0                    # resize scale factors
        bbox = [bbox[0] * sx, bbox[1] * sy, bbox[2] * sx, bbox[3] * sy]  # scale the box identically
        if self.augment and random.random() < 0.5:               # 50% horizontal flip: hands are
            img = img.transpose(Image.FLIP_LEFT_RIGHT)           # left/right symmetric so the label
            mask = mask.transpose(Image.FLIP_LEFT_RIGHT)         # is preserved
            x1, x2 = bbox[0], bbox[2]                            # mirror the x coordinates
            bbox[0], bbox[2] = IMG_SIZE - 1 - x2, IMG_SIZE - 1 - x1
        if self.augment:
            img = self.jitter(img)                               # photometric only -> geometry untouched
        img_t = torch.from_numpy(np.array(img)).permute(2, 0, 1).float() / 255.0  # HWC->CHW in [0,1]
        img_t = self.normalize(img_t)                            # ImageNet normalisation
        mask_t = (torch.from_numpy(np.array(mask)) > 0).float().unsqueeze(0)      # (1,H,W) binary
        return img_t, encode_yolo_target(bbox, item['cls']), mask_t

samples = scan_dataset(class_dirs)                               # build the sample list
train_idx, val_idx = split_per_class(samples, VAL_FRACTION, SEED)  # stratified split
train_ds = GestureDataset(samples, train_idx, augment=True)      # training set WITH augmentation
val_ds = GestureDataset(samples, val_idx, augment=False)         # val set WITHOUT (deterministic)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f'{len(samples)} samples -> {len(train_ds)} train / {len(val_ds)} val')


## Step 6 — sanity-check the labels (ALWAYS do this before training)
If the ground-truth box/mask drawn here does not sit on the hand, the annotation parsing is wrong and training would silently learn garbage — checking now costs seconds, debugging later costs hours.

In [ ]:
def denorm(img_t):
    # invert the ImageNet normalisation so matplotlib shows natural colours
    m = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)                # broadcastable mean
    s = torch.tensor(IMAGENET_STD).view(3, 1, 1)                 # broadcastable std
    return (img_t * s + m).clamp(0, 1).permute(1, 2, 0).numpy()  # CHW -> HWC for imshow

def target_to_box(t):
    # invert encode_yolo_target: grid target -> pixel corner box + class id
    obj = (t[..., 4] > 0.5).nonzero(as_tuple=False)[0]           # the single object cell (i,j)
    i, j = int(obj[0]), int(obj[1])
    cell = t[i, j]
    cx = (j + float(cell[0])) / S * IMG_SIZE                     # cell-relative -> pixels
    cy = (i + float(cell[1])) / S * IMG_SIZE
    bw, bh = float(cell[2]) * IMG_SIZE, float(cell[3]) * IMG_SIZE
    return [cx - bw / 2, cy - bh / 2, cx + bw / 2, cy + bh / 2], int(cell[5:].argmax())

fig, axes = plt.subplots(1, 4, figsize=(16, 4))                  # look at 4 random training samples
for ax in axes:
    img_t, tgt, mask_t = train_ds[random.randrange(len(train_ds))]  # random augmented sample
    box, cls = target_to_box(tgt)                                # decode the GT box back
    ax.imshow(denorm(img_t))                                     # the photo
    ax.imshow(mask_t[0].numpy(), alpha=0.35, cmap='Reds')        # GT mask overlay (red)
    ax.add_patch(plt.Rectangle((box[0], box[1]), box[2] - box[0], box[3] - box[1],
                               fill=False, edgecolor='lime', linewidth=2))  # GT box (green)
    ax.set_title(CLASS_NAMES[cls])                               # GT class name
    ax.axis('off')
plt.tight_layout(); plt.show()                                   # box+mask must sit ON the hand


## Step 7 — the model
**Why ResNet18 instead of the original Darknet-24?** The YOLOv1 paper pretrains Darknet-24 on ImageNet for about a week before detection training. With ~50 training images, a from-scratch backbone would memorise the training set instantly and fail on validation — so we keep YOLOv1's *detection formulation* (7x7 grid, 2 boxes/cell, per-cell class scores, the paper's loss) on top of an ImageNet-pretrained ResNet18.

**Why sigmoid on the detection output?** The paper used unbounded linear outputs; sigmoid bounds x, y, w, h, confidence and class scores to (0,1), which matches their meaning exactly and prevents exploding box predictions early in training — a real risk with so little data.

In [ ]:
def conv_block(in_ch, out_ch):
    # 3x3 conv + BN + LeakyReLU(0.1): YOLOv1's building block (LeakyReLU keeps a small
    # gradient for negative activations, which helps the regression outputs early on)
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),      # bias=False: BN has its own shift
        nn.BatchNorm2d(out_ch),                                  # BN stabilises small-batch training
        nn.LeakyReLU(0.1, inplace=True))

class YoloV1Gesture(nn.Module):
    def __init__(self):
        super().__init__()
        res = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)   # ImageNet weights: crucial for tiny data
        self.stem = nn.Sequential(res.conv1, res.bn1, res.relu, res.maxpool)  # output stride /4
        self.layer1 = res.layer1                                 # /4,  64ch (seg skip connection)
        self.layer2 = res.layer2                                 # /8, 128ch (seg skip connection)
        self.layer3 = res.layer3                                 # /16,256ch (seg skip connection)
        self.layer4 = res.layer4                                 # /32,512ch (semantic features)
        out_ch = B * 5 + C                                       # per-cell vector: B*(x,y,w,h,conf)+C
        self.det_head = nn.Sequential(
            nn.Conv2d(512, 512, 3, stride=2, padding=1, bias=False),  # 14x14 -> 7x7 = the SxS grid
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.1, inplace=True),
            conv_block(512, 256),                                # extra capacity like YOLO's last convs
            nn.Conv2d(256, out_ch, 1))                           # 1x1 conv -> raw per-cell predictions
        # U-Net-style decoder: upsample + fuse with the matching encoder stage. The skip
        # connections restore the spatial detail the encoder's downsampling threw away —
        # that is why this beats plain upsampling of the last feature map.
        self.up3 = conv_block(512 + 256, 256)                    # fuse upsampled /32 with /16
        self.up2 = conv_block(256 + 128, 128)                    # fuse with /8
        self.up1 = conv_block(128 + 64, 64)                      # fuse with /4
        self.seg_out = nn.Conv2d(64, 1, 1)                       # 1 channel: hand-mask logits
    def forward(self, x):
        x = self.stem(x)                                         # /4
        f1 = self.layer1(x)                                      # /4
        f2 = self.layer2(f1)                                     # /8
        f3 = self.layer3(f2)                                     # /16
        f4 = self.layer4(f3)                                     # /32
        det = torch.sigmoid(self.det_head(f4))                   # bound all outputs to (0,1), see above
        det = det.permute(0, 2, 3, 1)                            # (N,S,S,B*5+C): loss-friendly layout
        d3 = self.up3(torch.cat([F.interpolate(f4, size=f3.shape[2:], mode='bilinear', align_corners=False), f3], 1))
        d2 = self.up2(torch.cat([F.interpolate(d3, size=f2.shape[2:], mode='bilinear', align_corners=False), f2], 1))
        d1 = self.up1(torch.cat([F.interpolate(d2, size=f1.shape[2:], mode='bilinear', align_corners=False), f1], 1))
        seg = F.interpolate(self.seg_out(d1), size=(IMG_SIZE, IMG_SIZE), mode='bilinear', align_corners=False)
        return det, seg                                          # detection grid + full-res mask logits

model = YoloV1Gesture().to(DEVICE)                               # build and move to GPU
with torch.no_grad():                                            # quick shape smoke-test:
    d, s2 = model(torch.zeros(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE))
print('det:', tuple(d.shape), ' seg:', tuple(s2.shape))          # expect (1,7,7,20) and (1,1,448,448)


## Step 8 — the losses
The **YOLOv1 loss** (paper Eq. 3) is sum-squared error over 5 terms: box centre, box size (with `sqrt` so small boxes are not dominated by large ones), object confidence (target = IoU, per the paper), empty-cell confidence (damped by 0.5 because 48 of 49 cells are empty), and MSE classification (the paper's choice — cross-entropy only arrived in later YOLO versions). The *responsible* box of a cell is the one of its B boxes with highest IoU vs the ground truth — this specialisation is YOLOv1's core trick.

The **segmentation loss** is BCE + Dice: BCE gives smooth per-pixel gradients everywhere, Dice directly optimises overlap and is robust to the foreground/background imbalance (the hand covers a small fraction of the image, so BCE alone under-segments).

In [ ]:
def cell_to_xyxy(box, i, j):
    # (x_cell, y_cell, w, h) at grid cell (i,j) -> normalised corner coordinates
    cx = (j.float() + box[:, 0]) / S                             # cell-relative -> image-relative x
    cy = (i.float() + box[:, 1]) / S                             # cell-relative -> image-relative y
    w, h = box[:, 2], box[:, 3]                                  # already image-relative
    return torch.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], dim=1)

def iou_pairwise(a, b):
    # element-wise IoU between two (M,4) corner tensors (row i vs row i)
    ix1 = torch.max(a[:, 0], b[:, 0]); iy1 = torch.max(a[:, 1], b[:, 1])   # intersection corners
    ix2 = torch.min(a[:, 2], b[:, 2]); iy2 = torch.min(a[:, 3], b[:, 3])
    inter = (ix2 - ix1).clamp(min=0) * (iy2 - iy1).clamp(min=0)  # clamp: disjoint -> 0
    area_a = (a[:, 2] - a[:, 0]).clamp(min=0) * (a[:, 3] - a[:, 1]).clamp(min=0)
    area_b = (b[:, 2] - b[:, 0]).clamp(min=0) * (b[:, 3] - b[:, 1]).clamp(min=0)
    return inter / (area_a + area_b - inter + 1e-9)              # epsilon: no divide-by-zero

def yolo_v1_loss(pred, target):
    N = pred.shape[0]                                            # batch size for averaging
    boxes = pred[..., :B * 5].reshape(N, S, S, B, 5)             # split the B boxes per cell
    cls_pred = pred[..., B * 5:]                                 # per-cell class scores
    obj = target[..., 4] > 0.5                                   # object indicator per cell
    idx = obj.nonzero(as_tuple=False)                            # (M,3): batch, i, j of GT cells
    tb = target[..., :4][obj]                                    # GT boxes in cell format
    pb = boxes[obj]                                              # (M,B,5) predictions in GT cells
    gt_xyxy = cell_to_xyxy(tb, idx[:, 1], idx[:, 2])             # GT corners for IoU
    ious = torch.stack([iou_pairwise(cell_to_xyxy(pb[:, b_, :4], idx[:, 1], idx[:, 2]), gt_xyxy)
                        for b_ in range(B)], dim=1)              # IoU of each candidate box (M,B)
    best = ious.argmax(dim=1)                                    # index of the responsible box
    rng = torch.arange(pb.shape[0], device=pred.device)          # row selector
    resp = pb[rng, best]                                         # the responsible box vectors
    best_iou = ious[rng, best].detach()                          # confidence TARGET = IoU (paper);
    #                                                              detach: never backprop a target
    loss_xy = F.mse_loss(resp[:, 0:2], tb[:, 0:2], reduction='sum')          # term 1: centres
    loss_wh = F.mse_loss(torch.sqrt(resp[:, 2:4].clamp(min=1e-6)),           # term 2: sqrt sizes —
                         torch.sqrt(tb[:, 2:4].clamp(min=1e-6)), reduction='sum')  # equalises small/large
    loss_obj = F.mse_loss(resp[:, 4], best_iou, reduction='sum')             # term 3: object conf
    loss_noobj = (boxes[~obj][..., 4] ** 2).sum()                            # term 4: empty cells -> 0
    loss_cls = F.mse_loss(cls_pred[obj], target[..., 5:][obj], reduction='sum')  # term 5: classes
    return (LAMBDA_COORD * (loss_xy + loss_wh) + loss_obj
            + LAMBDA_NOOBJ * loss_noobj + loss_cls) / N          # weighted sum, batch-averaged

def seg_loss(logits, mask):
    bce = F.binary_cross_entropy_with_logits(logits, mask)       # on logits: numerically stabler
    prob = torch.sigmoid(logits)                                 # probabilities for Dice
    inter = (prob * mask).sum(dim=(1, 2, 3))                     # per-image intersection
    denom = prob.sum(dim=(1, 2, 3)) + mask.sum(dim=(1, 2, 3))    # per-image sums
    dice = 1 - ((2 * inter + 1) / (denom + 1)).mean()            # +1 smoothing avoids 0/0
    return bce + dice                                            # equal mix works well here


## Step 9 — (optional but recommended) single-batch overfit test
The classic debugging trick: a correct model + loss must be able to drive the loss near zero on ONE batch. If this cell's loss does not fall steeply, something is wired wrong — find out now, not after 60 epochs.

In [ ]:
dbg_model = YoloV1Gesture().to(DEVICE)                           # fresh model just for this test
dbg_opt = torch.optim.AdamW(dbg_model.parameters(), lr=1e-3)     # higher LR: we WANT to overfit
img_b, tgt_b, msk_b = next(iter(train_loader))                   # grab a single fixed batch
img_b, tgt_b, msk_b = img_b.to(DEVICE), tgt_b.to(DEVICE), msk_b.to(DEVICE)
dbg_model.train()
for step in range(60):                                           # 60 steps on the same batch
    det, seg = dbg_model(img_b)                                  # forward
    loss = yolo_v1_loss(det, tgt_b) + LAMBDA_SEG * seg_loss(seg, msk_b)  # combined loss
    dbg_opt.zero_grad(); loss.backward(); dbg_opt.step()         # standard update
    if step % 10 == 0:
        print(f'step {step:2d}  loss {loss.item():.3f}')         # should drop steeply
del dbg_model, dbg_opt; torch.cuda.empty_cache()                 # free the debug model's GPU memory


## Step 10 — training
**AdamW + cosine schedule instead of the paper's SGD + hand-tuned schedule**: AdamW adapts per-parameter step sizes, converging much faster on small datasets with no warmup tuning; cosine annealing decays the LR smoothly to ~0 without having to pick step milestones. The checkpoint with the **best validation box IoU** is kept — localisation is the hardest task here, so it discriminates checkpoints better than accuracy (which saturates early).

In [ ]:
def decode_predictions(pred):
    # one raw output (S,S,B*5+C) -> (class_id, score, [x1,y1,x2,y2] pixels). We keep only the
    # single best box (confidence x class prob) because every image has exactly one gesture —
    # simpler and more accurate than generic NMS for this dataset.
    boxes = pred[..., :B * 5].reshape(S, S, B, 5)                # split the B boxes
    cls_prob = pred[..., B * 5:]                                 # class probabilities per cell
    conf = boxes[..., 4]                                         # objectness per box
    best_cls_p, best_cls = cls_prob.max(dim=-1)                  # best class per cell
    scores = conf * best_cls_p.unsqueeze(-1)                     # YOLO score = conf * class prob
    flat = scores.flatten().argmax()                             # overall best box
    i = int(flat // (S * B)); j = int((flat % (S * B)) // B); b_ = int(flat % B)  # unflatten (i,j,b)
    bx = boxes[i, j, b_]                                         # winning box vector
    cx = (j + bx[0].item()) / S * IMG_SIZE                       # cell-relative -> pixel centre
    cy = (i + bx[1].item()) / S * IMG_SIZE
    bw, bh = bx[2].item() * IMG_SIZE, bx[3].item() * IMG_SIZE    # image-relative -> pixels
    box = [max(0, cx - bw / 2), max(0, cy - bh / 2),             # centre -> clipped corners
           min(IMG_SIZE - 1, cx + bw / 2), min(IMG_SIZE - 1, cy + bh / 2)]
    return int(best_cls[i, j]), float(scores[i, j, b_]), box

@torch.no_grad()                                                 # evaluation needs no gradients
def evaluate(model, loader):
    model.eval()                                                 # freeze BN statistics
    tot, n = 0.0, 0
    correct, bious, mious = 0, [], []
    for img, tgt, msk in loader:
        img, tgt, msk = img.to(DEVICE), tgt.to(DEVICE), msk.to(DEVICE)
        det, seg = model(img)                                    # forward
        tot += (yolo_v1_loss(det, tgt) + LAMBDA_SEG * seg_loss(seg, msk)).item() * img.shape[0]
        prob = torch.sigmoid(seg)                                # mask probabilities
        for k in range(img.shape[0]):                            # per-image metrics
            n += 1
            cp, _, pbox = decode_predictions(det[k].cpu())       # predicted class + box
            gbox, gc = target_to_box(tgt[k].cpu())               # GT class + box (from Step 6 helper)
            correct += int(cp == gc)                             # classification accuracy
            bious.append(iou_xyxy(pbox, gbox))                   # box IoU
            pm = (prob[k, 0] > 0.5).float()                      # binarised predicted mask
            gm = msk[k, 0]                                       # GT mask
            inter = (pm * gm).sum().item()                       # mask IoU numerator
            union = pm.sum().item() + gm.sum().item() - inter    # mask IoU denominator
            mious.append(inter / union if union > 0 else 1.0)    # both empty -> perfect
    return tot / max(n, 1), correct / max(n, 1), float(np.mean(bious)), float(np.mean(mious))

model = YoloV1Gesture().to(DEVICE)                               # fresh model for the real run
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
history = {'train': [], 'val': [], 'acc': [], 'biou': [], 'miou': []}  # for the curves
best_iou = 0.0                                                   # best val box IoU so far
for epoch in tqdm(range(1, EPOCHS + 1), desc='epochs'):
    model.train()                                                # training mode (BN updates, augment)
    running = 0.0
    for img, tgt, msk in train_loader:                           # one pass over the training data
        img, tgt, msk = img.to(DEVICE), tgt.to(DEVICE), msk.to(DEVICE)
        det, seg = model(img)                                    # forward through both heads
        loss = yolo_v1_loss(det, tgt) + LAMBDA_SEG * seg_loss(seg, msk)  # multi-task loss
        optimizer.zero_grad()                                    # clear old gradients
        loss.backward()                                          # backprop through both heads
        optimizer.step()                                         # update all weights
        running += loss.item() * img.shape[0]
    scheduler.step()                                             # cosine LR decay
    vl, acc, biou, miou = evaluate(model, val_loader)            # validate every epoch (cheap here)
    history['train'].append(running / len(train_ds)); history['val'].append(vl)
    history['acc'].append(acc); history['biou'].append(biou); history['miou'].append(miou)
    if biou > best_iou:                                          # keep only the best checkpoint
        best_iou = biou
        torch.save({'model': model.state_dict(), 'classes': CLASS_NAMES}, 'best_model.pt')
    if epoch % 5 == 0 or epoch == 1:                             # print every 5 epochs to keep logs short
        print(f'epoch {epoch:3d} | train {history["train"][-1]:.3f} | val {vl:.3f} '
              f'| acc {acc:.2f} | box IoU {biou:.3f} | mask IoU {miou:.3f}')
print(f'best val box IoU: {best_iou:.3f} -> best_model.pt')


In [ ]:
# ---------------- training curves: quick visual health-check of the run ----------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train'], label='train loss')               # falling train loss = learning
axes[0].plot(history['val'], label='val loss')                   # gap to val loss = overfitting amount
axes[0].set_xlabel('epoch'); axes[0].legend(); axes[0].set_title('loss')
axes[1].plot(history['acc'], label='class accuracy')             # the three quality metrics
axes[1].plot(history['biou'], label='box IoU')
axes[1].plot(history['miou'], label='mask IoU')
axes[1].set_xlabel('epoch'); axes[1].legend(); axes[1].set_title('validation metrics')
plt.tight_layout(); plt.show()


## Step 11 — final evaluation + qualitative results
Reload the **best** checkpoint (the last epoch is not necessarily the best one) and visualise predictions on validation images: green = predicted box, dashed white = ground-truth box, red = predicted mask.

In [ ]:
ckpt = torch.load('best_model.pt', map_location=DEVICE)          # load the best checkpoint
model.load_state_dict(ckpt['model'])                             # restore its weights
vl, acc, biou, miou = evaluate(model, val_loader)                # final numbers on validation
print(f'FINAL  val loss {vl:.3f} | class accuracy {acc:.2%} | box IoU {biou:.3f} | mask IoU {miou:.3f}')

model.eval()                                                     # deterministic inference
n_show = min(8, len(val_ds))                                     # up to 8 validation images
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
with torch.no_grad():
    for ax, k in zip(axes.flatten(), range(n_show)):
        img_t, tgt, msk = val_ds[k]                              # one validation sample
        det, seg = model(img_t.unsqueeze(0).to(DEVICE))          # forward a single image
        cp, score, pbox = decode_predictions(det[0].cpu())       # predicted class/score/box
        pmask = (torch.sigmoid(seg[0, 0]).cpu().numpy() > 0.5)   # predicted binary mask
        gbox, gc = target_to_box(tgt)                            # ground truth for comparison
        ax.imshow(denorm(img_t))                                 # the photo
        ax.imshow(pmask, alpha=0.35, cmap='Reds')                # predicted mask (red overlay)
        ax.add_patch(plt.Rectangle((pbox[0], pbox[1]), pbox[2] - pbox[0], pbox[3] - pbox[1],
                                   fill=False, edgecolor='lime', linewidth=2))     # prediction
        ax.add_patch(plt.Rectangle((gbox[0], gbox[1]), gbox[2] - gbox[0], gbox[3] - gbox[1],
                                   fill=False, edgecolor='white', linewidth=1, linestyle='--'))  # GT
        ok = 'OK' if cp == gc else 'WRONG'                       # flag misclassifications
        ax.set_title(f'pred {CLASS_NAMES[cp]} ({score:.2f}) | gt {CLASS_NAMES[gc]} [{ok}]', fontsize=9)
        ax.axis('off')
for ax in axes.flatten()[n_show:]:                               # hide unused subplot axes
    ax.axis('off')
plt.tight_layout(); plt.show()


In [ ]:
# ---------------- download the trained weights (optional) ----------------
try:
    from google.colab import files                               # Colab-only download helper
    files.download('best_model.pt')                              # save the checkpoint to your machine
except ImportError:
    print('best_model.pt saved in the working directory')        # local run: file is already there


## Notes, limitations and extensions
* **Tiny dataset (~6 images/class):** expect the metrics to vary a few points between runs; the pretrained backbone + augmentation + weight decay are what make learning possible at all. More data is the single most effective improvement.
* **Depth channel:** to use the `depth` folder, change the stem conv to 4 input channels (copy the pretrained RGB kernels and initialise the 4th channel to their mean) and stack depth onto the input in `__getitem__`. Left out here because pretrained RGB features outweigh a from-scratch 4th channel on this little data.
* **YOLOv1 known weaknesses** (from the paper) that apply here too: coarse 7x7 grid limits localisation precision, and at most one class per cell — irrelevant for single-gesture images, important if you ever have two hands close together.
* **If annotations are boxes only** (txt/JSON without polygons), the segmentation target degrades to a rectangle; the seg head then learns box-shaped masks — the detection task is unaffected.